# Shelf List Report

This notebook will create a CSV file of Inventory Items with the following information: Barcode, Title, Effective Location, Effective Call Number Components, Material Type, and Item Status

## 1. Environment setup

In [98]:
# This script will import the following Python libraries, but not install them. If you are missing any of these, you can install them via the command line like this:
# !pip install pandas
import pandas as pd
import requests
from datetime import datetime

pd.set_option('display.max_columns', None)

## 2. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [99]:
%run folio_auth.ipynb

Login succeeded. Token retrieved.


## 3. Helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. 

In [100]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    return all_records

## 4. Get Locations

In [ ]:
locs_raw = fetch_all_records(
    "/locations",
    records_key="locations",
    limit=50,
    query='isActive=="true"',
) 
print(f"{len(locs_raw)} active locations found")
locs_df = pd.DataFrame(locs_raw)
# locs_df.sort_values(by='name',inplace=True)
print(locs_df.loc[:, ['name','id']].sort_values(by='name'))


27 active locations found
                                  name                                    id
0               Music Library Reserves  757c6f19-bf53-4c63-b314-135e7360afc0
1                   Interlibrary Loans  b8b8dcd5-69f3-4165-909e-435e8b9cdd2e
2                      Suggestion Form  923b821c-a192-43b6-b1a7-8e03f50d1086
3     Main Library Special Collections  8ee52ff5-d149-48bc-bf4f-749a0097325e
4                      Law Main Stacks  e3775cef-9d30-4896-a452-98a099cf9bfa
5      Main Library Technical Services  d9d54cc4-b052-42b8-8dd2-0f192752d88d
6                    MC Alton Reserves  4df811f6-2c1b-4690-aa08-66a12d5f3bf3
7                   MC Alton Reference  7bd4f123-cacd-458b-acbb-458a82f9f34c
8                 MC Alton Periodicals  ecc6b76e-4b6a-49de-8378-d7edd8ba3e20
9                          Bus Lib Ref  aa902fd1-ca2e-4d06-89ab-96f79e1feee0
10                     MC Alton Stacks  6ecf014b-1643-4fb9-839c-8a271453f72e
11                        MED Reserves  9f839129-4

## 5. Retrieve Item records with that Effective Location

In [107]:
location_id = 'fc4b17f0-3745-4827-8e74-08f004003f7d' # Med Director's Office
# location_id = '4df811f6-2c1b-4690-aa08-66a12d5f3bf3' # MC Alton Reserves
# location_id = '51dfa4a9-841b-47e1-9d71-eb2f8cf3176b' # Main Stacks
# location_id = 'REPLACE WITH THE LOCATION ID FROM ABOVE'

# df.loc[df['column_name'] == value]
search_loc = locs_df.loc[locs_df['id'] == location_id]


items_raw = fetch_all_records(
    "/inventory/items",
    records_key="items",
    query='effectiveLocationId=='+location_id
) 
print(f"{len(items_raw)} items found for with an effective location of: {search_loc['name']}")
items_df = pd.DataFrame(items_raw)

items_df.head()


10 items found for with an effective location of: 24    MED Director's Office
Name: name, dtype: str


,id,_version,status,administrativeNotes,title,callNumber,hrid,contributorNames,formerIds,discoverySuppress,holdingsRecordId,barcode,notes,circulationNotes,tags,yearCaption,electronicAccess,statisticalCodeIds,purchaseOrderLineIdentifier,materialType,permanentLoanType,metadata,effectiveCallNumberComponents,effectiveShelvingOrder,isBoundWith,effectiveLocation
0,d9295ee7-ffbc-4b55-8e70-0644434dc887,6,"{'name': 'Available', 'date': '2022-06-02T21:5...",[],Atlas of radiologic measurement / Theodore E. ...,RC78 .K315 2001,it00000256610,"[{'name': 'Keats, Theodore E. (Theodore Eliot)...",[],None,f608af3f-aa88-4fd1-a4a0-f9b9734ffb56,32260007476429,[],[],{'tagList': []},[],[],[],None,"{'id': '13493ed9-900c-4db2-8646-4798bd8ac213',...","{'id': '2b94c631-fca9-4892-a730-03ee529ffe27',...",{'createdDate': '2022-06-02T21:53:47.748+00:00...,"{'callNumber': 'RC78 .K315 2001 ', 'prefix': N...",RC 278 K315 42001,False,"{'id': 'fc4b17f0-3745-4827-8e74-08f004003f7d',..."
1,939189da-d369-4805-b6cd-000c04f2b298,5,"{'name': 'Available', 'date': '2022-06-02T21:5...",[],Health sciences collection management for the ...,RT48 .R48 2018,it00000261406,"[{'name': 'Kendall, Susan K., 1968-'}]",[],None,0b58df88-22f9-494d-80ce-39bfde1dc0f9,32260010683078,[],[],{'tagList': []},[],[],[],None,"{'id': '13493ed9-900c-4db2-8646-4798bd8ac213',...","{'id': '2b94c631-fca9-4892-a730-03ee529ffe27',...",{'createdDate': '2022-06-02T21:58:48.574+00:00...,"{'callNumber': 'RT48 .R48 2018', 'prefix': Non...",RT 248 R48 42018,False,"{'id': 'fc4b17f0-3745-4827-8e74-08f004003f7d',..."
2,f2f2cceb-1fbc-4550-8c95-afdf4908b4ad,6,"{'name': 'Available', 'date': '2022-06-02T13:2...",[],Management accounting in health care organizat...,RA971.3 .Y68 2003,it00000196065,"[{'name': 'Young, David W'}]",[],None,f49559b0-8ef1-411b-9b90-3fde7bd1905e,32260005323037,[],[],{'tagList': []},[],[],[],None,"{'id': '13493ed9-900c-4db2-8646-4798bd8ac213',...","{'id': '2b94c631-fca9-4892-a730-03ee529ffe27',...",{'createdDate': '2022-06-02T13:24:15.692+00:00...,"{'callNumber': 'RA971.3 .Y68 2003 ', 'prefix':...",RA 3971.3 Y68 42003,False,"{'id': 'fc4b17f0-3745-4827-8e74-08f004003f7d',..."
3,86fd42bd-1037-4590-a70b-2917e81f4c15,6,"{'name': 'Available', 'date': '2022-06-02T15:2...",[],Financial management for health care instituti...,RA971.3 .F87,it00000224877,"[{'name': 'Furst, Richard W'}]",[],None,ecbabe07-8cbf-41d4-a646-5f10c3780d34,32260008276794,[],[],{'tagList': []},[],[],[],None,"{'id': '13493ed9-900c-4db2-8646-4798bd8ac213',...","{'id': '2b94c631-fca9-4892-a730-03ee529ffe27',...",{'createdDate': '2022-06-02T15:28:20.410+00:00...,"{'callNumber': 'RA971.3 .F87 ', 'prefix': None...",RA 3971.3 F87,False,"{'id': 'fc4b17f0-3745-4827-8e74-08f004003f7d',..."
4,8aa3c091-2fbe-42ce-9a78-035a01bca645,6,"{'name': 'Available', 'date': '2022-05-10T02:1...",[],Health care quality & outcomes management / He...,RA399.A1 G855 1999,it00000075554,[{'name': 'Health and Administration Developme...,[],None,16b5c5b8-e89e-4325-a035-1df20b7657b7,32260008344543,[],[],{'tagList': []},[],[],[],None,"{'id': '13493ed9-900c-4db2-8646-4798bd8ac213',...","{'id': '2b94c631-fca9-4892-a730-03ee529ffe27',...",{'createdDate': '2022-05-10T02:13:11.785+00:00...,"{'callNumber': 'RA399.A1 G855 1999 ', 'prefix'...",RA 3399 A1 G855 41999,False,"{'id': 'fc4b17f0-3745-4827-8e74-08f004003f7d',..."


## 9. Simplify Your Data
A combined dataframe will include many columns that you won't want. You can create a new dataframe by extracting only the columns you want into a new set. 

,barcode,prefix,callNumber,suffix,materialTypeId,status,title
0,32260007672449,NaN,P118 .D444 1999,NaN,Book,Available,The development of language / edited by Martyn...
1,32260007145701,NaN,HV6046 .S54 2005,NaN,Book,Available,The crimes women commit : the punishments they...
2,NaN,NaN,TL560.1.A736 2021,NaN,Book,On order,Aviation high school student notebook : learn ...
3,32260010723346,NaN,BX1749 .P4 1492,NaN,Book,Available,Textus Sententiaru[m] cum conclusionib[us] Hen...
4,32260001344425,NaN,PQ2344.Z5 P38,NaN,Book,Available,The development of Mallarmé's prose style. / W...


In [62]:
flattened_items = pd.DataFrame()
flattened_items['barcode'] = items_df['barcode']
flattened_items['prefix'] = items_df['effectiveCallNumberComponents'].apply(lambda x: x.get('prefix') if isinstance(x,dict) else None)
flattened_items['callNumber'] = items_df['effectiveCallNumberComponents'].apply(lambda x: x.get('callNumber') if isinstance(x,dict) else None)
flattened_items['suffix'] = items_df['effectiveCallNumberComponents'].apply(lambda x: x.get('suffix') if isinstance(x,dict) else None)
flattened_items['materialTypeId']  =  items_df['materialType'].apply(lambda x: x.get('name') if isinstance(x,dict) else None)
flattened_items['status']  =  items_df['status'].apply(lambda x: x.get('name') if isinstance(x,dict) else None)
flattened_items['title'] = items_df['title']

flattened_items.head()

,barcode,prefix,callNumber,suffix,materialTypeId,status,title
0,32260007672449,NaN,P118 .D444 1999,NaN,Book,Available,The development of language / edited by Martyn...
1,32260007145701,NaN,HV6046 .S54 2005,NaN,Book,Available,The crimes women commit : the punishments they...
2,NaN,NaN,TL560.1.A736 2021,NaN,Book,On order,Aviation high school student notebook : learn ...
3,32260010723346,NaN,BX1749 .P4 1492,NaN,Book,Available,Textus Sententiaru[m] cum conclusionib[us] Hen...
4,32260001344425,NaN,PQ2344.Z5 P38,NaN,Book,Available,The development of Mallarmé's prose style. / W...


In [ ]:
# date = datetime.now("%Y-%m-%d")
# # date = str(date("%Y-%m-%d"))
# print(date)

today = date.today().strftime("%Y-%m-%d")
print(today)

flattened_items.to_csv(today + 'Instances Created since ' + str(search_date) + '.csv', index=False)

2026-08-20
